# CofC 2026 Match Intake

Use this notebook after each match to inspect the original Wyscout exports and create a review bundle. **It does not publish to Supabase or change COUG scores.** Keep vendor filenames unchanged.

Workflow: mount Drive → enter match details → inspect → review warnings → create the bundle → send it for staff approval.

## 1. Connect Google Drive

If you are running locally instead of Colab, this cell safely does nothing.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local Jupyter session detected; Google Drive mount skipped.')

## 2. Match setup

Change only the values in this cell. `SOURCE_FOLDER` must contain the untouched files downloaded from Wyscout. `OUTPUT_FOLDER` must be a different folder.

In [ ]:
SEASON = '2026'
MATCH_SLUG = '2026-08-20_davidson'
SOURCE_FOLDER = '/content/drive/MyDrive/CofC Soccer Analytics/2026/matches/2026-08-20_davidson/00_source'
OUTPUT_FOLDER = '/content/drive/MyDrive/CofC Soccer Analytics/2026/matches/2026-08-20_davidson/20_generated'

MATCH_METADATA = {
    'match_date': '2026-08-20',
    'opponent': 'Davidson',
    'location': 'home',  # home, away, or neutral
    'competition': 'non-conference',
    'cofc_score': None,
    'opponent_score': None,
    'prepared_by': '',
    'notes': '',
}

# Leave False for the first run. Change to True only after reviewing inspection results.
CREATE_REVIEW_BUNDLE = False

## 3. Load the tested pipeline

Colab downloads the current repository. A local notebook uses the existing checkout. No secrets are requested.

In [ ]:
import subprocess
import sys
import importlib.util

REPO_URL = 'https://github.com/anissawilliams/cofc-soccer-analytics.git'
if IN_COLAB:
    REPO_ROOT = Path('/content/cofc-soccer-analytics')
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((path for path in candidates if (path / 'pipeline').is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError('Open this notebook from the cofc-soccer-analytics repository.')

if importlib.util.find_spec('pypdf') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf>=5.0.0'], check=True)

INTAKE_SCRIPT = REPO_ROOT / 'pipeline' / 'ingestion' / 'prepare_match_intake.py'
ROSTER = REPO_ROOT / 'pipeline' / 'ingestion' / f'roster_{SEASON}.csv'
print(f'Pipeline: {INTAKE_SCRIPT}')
print(f'Roster:   {ROSTER}')

## 4. Inspect the source files

This is a dry run: it reads the files but writes nothing. Green means that output can be prepared for staff review—not that it has been published.

In [ ]:
import json
import pandas as pd
from IPython.display import display

source_dir = Path(SOURCE_FOLDER).expanduser()
output_dir = Path(OUTPUT_FOLDER).expanduser()
if not source_dir.is_dir():
    raise FileNotFoundError(f'Source folder does not exist: {source_dir}')
if source_dir.resolve() == output_dir.resolve() or source_dir.resolve() in output_dir.resolve().parents:
    raise ValueError('OUTPUT_FOLDER must not be inside SOURCE_FOLDER. Keep source files untouched.')
if not ROSTER.is_file():
    raise FileNotFoundError(f'Roster not found for season {SEASON}: {ROSTER}')
required_metadata = ['match_date', 'opponent', 'location', 'prepared_by']
missing_metadata = [key for key in required_metadata if not str(MATCH_METADATA.get(key) or '').strip()]
if missing_metadata:
    raise ValueError(f'Complete these MATCH_METADATA fields: {missing_metadata}')
if MATCH_METADATA['location'] not in {'home', 'away', 'neutral'}:
    raise ValueError("MATCH_METADATA location must be 'home', 'away', or 'neutral'.")
if not MATCH_SLUG.startswith(str(MATCH_METADATA['match_date'])):
    raise ValueError('MATCH_SLUG and MATCH_METADATA match_date do not agree.')
if not str(MATCH_METADATA['match_date']).startswith(str(SEASON)):
    raise ValueError('SEASON and MATCH_METADATA match_date do not agree.')

command = [
    sys.executable, str(INTAKE_SCRIPT),
    '--input-dir', str(source_dir),
    '--season', SEASON,
    '--slug', MATCH_SLUG,
    '--roster', str(ROSTER),
    '--dry-run',
]
result = subprocess.run(command, check=True, capture_output=True, text=True)
report = json.loads(result.stdout)

readiness = pd.DataFrame([
    {'Use': 'Match analytics', 'Ready': report['analytics']['ready'], 'Reason': report['analytics']['reason']},
    {'Use': 'COUG player scoring', 'Ready': report['scoring']['ready'], 'Reason': report['scoring']['reason']},
    {'Use': 'Official minutes/lineups', 'Ready': report['minutes']['ready'], 'Reason': report['minutes']['reason']},
])
display(readiness.style.map(lambda value: 'background-color: #d7f5df' if value is True else ('background-color: #fff1c7' if value is False else '')))
display(pd.DataFrame(report['files']))
print(f"Source files inventoried: {len(report.get('source_manifest') or [])}")
if report.get('scoring', {}).get('selected_file'):
    print('COUG scoring source:', report['scoring']['selected_file'])
if (report.get('team_event_summary') or {}).get('unmapped_labels'):
    print('REVIEW REQUIRED — unmapped labels:', report['team_event_summary']['unmapped_labels'])

## 5. Review before continuing

Check the table above. Missing COUG scoring usually means the player-coded Sportscode XML was not included; team-event XML files alone cannot produce player scores. Missing official minutes/lineups means the media-team final box score PDF is not in `00_source/official`. Unknown or unreadable XML files must be escalated.

When the inspection looks correct, return to the setup cell, change `CREATE_REVIEW_BUNDLE` to `True`, and run the remaining cells.

In [ ]:
if not CREATE_REVIEW_BUNDLE:
    raise RuntimeError('Inspection complete. Review the results, then set CREATE_REVIEW_BUNDLE = True in the setup cell.')

output_dir.mkdir(parents=True, exist_ok=True)
metadata_path = output_dir / f'{MATCH_SLUG}_metadata.json'
metadata_path.write_text(json.dumps(MATCH_METADATA, indent=2), encoding='utf-8')

prepare_command = [
    sys.executable, str(INTAKE_SCRIPT),
    '--input-dir', str(source_dir),
    '--output-dir', str(output_dir),
    '--season', SEASON,
    '--slug', MATCH_SLUG,
    '--roster', str(ROSTER),
    '--metadata', str(metadata_path),
]
subprocess.run(prepare_command, check=True)
final_report = json.loads((output_dir / f'{MATCH_SLUG}_intake_report.json').read_text())
print(f"Review status: {final_report['validation']['status']}")
print(f'Review bundle: {output_dir}')
print('Nothing has been published. Send the validation report and bundle for staff approval.')

## Done

Send the generated validation report to the staff reviewer. Do not edit files in the source folder and do not load anything into production yourself.